## Instalar los requiriments


In [ ]:
uv venv
uv pip install -r requirements.txt

## Librerías y constantes

In [ ]:
import json, os, re
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import copy

USE_REAL_API = True          
ENFERMEDAD = "esquizofrenia"           
BASE = "https://cima.aemps.es/cima/rest"
OUTPUT_FOLDER = Path("salidas"); OUTPUT_FOLDER.mkdir(exist_ok=True)
DATA_FOLDER = Path("data")

## Obtener la información de cómo están repartidos los números de registro y obtener los números de registro
1. Realizo una petición post con lo que pide el enunciado HU1.
2. Obtengo el número de filas y el tamaño de página (filas por página). Realizo una división entera y sumo 1 al resultado para aproxinar al siguiente número entero.
3. Realizo tantas peticiones como páginas tenga que recorrer, pasando la página como argumento.
4. Del resultado (``data = res.json()``), obtengo la lista de diccionarios ``data["resultados"]``, y para cada resultado, ``registro["nregistro"]``. Apendo cada número de registros a la lista de números de registro. 

In [18]:
response = requests.post(
    url=BASE+"/buscarEnFichaTecnica",
    json=[{ 
          "seccion": "4.1",
          "texto": ENFERMEDAD,
          "contiene": 1 
    }]
)

data = response.json()
total_filas = data['totalFilas']
tam_pagina = data['tamanioPagina']
num_pags = total_filas // tam_pagina + 1
nregistros = []
for i in range(1, num_pags+1):
    res = requests.post(
        url= BASE + "/buscarEnFichaTecnica",
        json=[{ 
            "seccion": "4.1",
            "texto": ENFERMEDAD,
            "contiene": 1 
        }],
        params={
            "pagina": i
        }
    )
    if res.status_code != 200:
        break
    data = res.json()
    for registro in data["resultados"]:
        nregistros.append(registro["nregistro"])
        

## Función auxiliar para obtener la url de la foto de los materiales
Dada una lista de diccionarios ``foto``, si el diccionario tiene el campo ``tipo == "materialas"`` y el campo ``url``, se devuelve la ``url``, si no, ``NA``.

In [19]:
# Está mal la API, "materiales" está escrito como "materialas"
def foto_materiales(listaFotos):
    for foto in listaFotos:
        if "tipo" in foto and foto["tipo"] == "materialas" and "url" in foto:
            return foto["url"]
        
    return pd.NA    

## Comprobación intermedia de qué devuelve la API para un número de registro cualquiera

In [20]:
res = requests.get(
    url=BASE + "/medicamento",
    params={
        "nregistro": nregistros[5]
    }
)
data = res.json()

print(json.dumps(data, indent=2))

{
  "nregistro": "79317",
  "nombre": "ARIPIPRAZOL CINFA 10 MG COMPRIMIDOS EFG",
  "pactivos": "ARIPIPRAZOL",
  "labtitular": "Laboratorios Cinfa S.A.",
  "labcomercializador": "Laboratorios Cinfa S.A.",
  "cpresc": "Medicamento Sujeto A Prescripci\u00f3n M\u00e9dica",
  "estado": {
    "aut": 1418133143000
  },
  "comerc": true,
  "receta": true,
  "generico": true,
  "conduc": true,
  "triangulo": false,
  "huerfano": false,
  "biosimilar": false,
  "nosustituible": {
    "id": 0,
    "nombre": "N/A"
  },
  "psum": false,
  "notas": false,
  "materialesInf": false,
  "ema": false,
  "docs": [
    {
      "tipo": 1,
      "url": "https://cima.aemps.es/cima/pdfs/ft/79317/FT_79317.pdf",
      "urlHtml": "https://cima.aemps.es/cima/dochtml/ft/79317/FT_79317.html",
      "secc": true,
      "fecha": 1770247804000
    },
    {
      "tipo": 2,
      "url": "https://cima.aemps.es/cima/pdfs/p/79317/P_79317.pdf",
      "urlHtml": "https://cima.aemps.es/cima/dochtml/p/79317/P_79317.html",
    

## Obtención de los datos de los medicamentos
Dada una lista de registros ``nregistros``, se recorre la lista y se hace una petición ``GET`` a ``BASE + "/medicamento"``, pasando como parámetro el ``nregistro``.

1. Se realiza la petición.
2. Si falla, se termina y se vacía la lista de diccionarios tipo ``medicamento``.
3. Se parsea la respuesta a ``json``.
4. Se crea un diccionario ``medicamento`` con los campos pertinentes, asegurando que si el valor correspondiente que viene en el json no aparece, la función **no falle sino, que rellene el valor con ``pd.NA``** o ``0`` en campos de longitud.
5. Se apenda el diccionario ``medicamento`` a la lista ``medicamentos``.
6. Se exporta el resultado a un archivo ``.xlsx``.

In [21]:
medicamentos = []
for i in nregistros:
    res = requests.get(
        url=BASE + "/medicamento",
        params={
            "nregistro": i
        }
    )
    data = res.json()
    medicamento = {
        "nregistro": data["nregistro"] if "nregistro" in data else pd.NA,
        "nombre": data["nombre"] if "nombre" in data else pd.NA,
        "pactivos": data["pactivos"] if "pactivos" in data else pd.NA,
        "labtitular": data["labtitular"] if "labtitular" in data else pd.NA,
        "labcomercializador": data["labcomercializador"] if "labcomercializador" in data else pd.NA,
        "cn": data["presentaciones"][0]["cn"] if "presentaciones" in data and len(data["presentaciones"]) > 0 and "cn" in data["presentaciones"][0] else pd.NA,
        "pactivos": (data["principiosActivos"][0]["cantidad"] + data["principiosActivos"][0]["unidad"]) if "principiosActivos" in data and len(data["principiosActivos"]) > 0 and "cantidad" in data["principiosActivos"][0] and "unidad" in data["principiosActivos"][0] else pd.NA,
        "forma_farmaceutica_simplificada": data["formaFarmaceuticaSimplificada"]["nombre"] if "formaFarmaceuticaSimplificada" in data and "nombre" in data["formaFarmaceuticaSimplificada"] else pd.NA,
        "estado_aut": pd.to_datetime(data["estado"]["aut"], unit="ms") if "estado" in data and "aut" in data["estado"] else pd.NA,
        "estado_rev": data["estado"]["rev"] if "estado" in data and "rev" in data["estado"] else pd.NA,
        "vias_administracion": 1 if "comerc" in data and data["comerc"] == True else 0,
        "requiere_receta": 1 if "receta" in data and data["receta"] == True else 0,
        "afecta_conduccion": 1 if "conduc" in data and data["conduc"] == True else 0,
        "triangulo_negro": 1 if "triangulo" in data and data["triangulo"] == True else 0,
        "medicamento_huerfano": 1 if "huerfano" in data and data["huerfano"] == True else 0,
        "biosimilar": 1 if "biosimilar" in data and data["biosimilar"] == True else 0,
        "url_html_ficha_tecnica": data["docs"][0]["urlHtml"] if "docs" in data else 0,
        "url_foto_materiales": foto_materiales(data["fotos"]) if "fotos" in data else pd.NA,
        "num_registros_atc": len(data["atcs"]) if "atcs" in data else 0,
        "num_principios_activos": len(data["pactivos"]) if "pactivos" in data else 0,
        "num_excipientes": len(data["excipientes"]) if "excipientes" in data else 0,
    }
    
    medicamentos.append(medicamento)
 
    if res.status_code != 200:
        medicamentos = []
        break
    
if len(medicamentos) > 0:
    df = pd.DataFrame(
        data=medicamentos
    )
    df.to_excel(OUTPUT_FOLDER / "medicamentos.xlsx", index=False)
    

## Datos agregados por web scraping
Se realiza un excel resultado de agregar datos al anterior `df` gracias al web scraping.

**Paso 0:** comprobar el html para una petición y ver cómo está estructurado y dónde se encuentran las etiquetas que nos interesan. Para ello, hacemos una petición, convertimos el contenido a ``BeautifulSoup`` y hacemos ``print(soup)``.

1. En ``medicamentos_agregados``, se crea una copia exacta de la lista de diccionarios ``medicamento``. De esta manera, si falla alguna petición, la lista ``medicamentos`` está intacta.
2. Para cada medicamento, si tiene ``url_html``, realizo una petición ``GET`` a dicha url.
3. Si falla alguna, vacío lista de agregados y termino.
4. Si no, creo un objeto soup con el contenido de la respuesta ``soup = BeautifulSoup(res.content, "html.parser")``.
5. Extracción de información:
   1. ``seccion4_4 = soup.find(id="4.4")``: Me devuelve el ``Tag`` con ``id="4.4"``.
      1. Como hice el paso 0, sé que el contenido de la sección **4.4** está en el ``<div>`` siguiente a ``section4_4``. Hay varias formas de obtener el ``<div>``, pero yo lo hice con ``div = seccion4_4.find_next()``. 
      2. Ahora, hay que extraer todo el texto contenido en ``div``. Lo obtengo con ``contenido = div.get_text(" ", strip=True)``. Lo que hace ``strip=True`` es eliminar los ``\n``, por ejemplo ``<p> Texto.</p>\n`` sería  ``Texto.\n.`` sin ``strip=True``. El primer parámetro ``" "`` hace que ``<p> Texto.</p>\n <p>Texto 2.</p>`` sea ``Texto. Texto 2.`` y no ``Texto.Texto 2.``.
      3. Con ``len(contenido.split(" "))``, divido todo el texto usando ``" "`` como separador. Podría almacenar el texto en una lista de palabras, pero me lo ahorro porque no lo quiero. Con ``len(lista)`` obtengo el tamaño de la lista, que es el número de palabras.
   2. ``len(soup.find_all("table"))``: ``find_all(Tag)`` devuelve una lista con todas las Tag del tipo que se haya indicado. Si no hay ninguna, devuelve ``[]``.
   3. Número de grave/graves.
      1. ``texto_html = soup.get_text(" ", strip=True)``: Obtengo todo el texto, realmente ``" "`` no es necesario aquí.
      2. ``len(re.findall(r"\bgraves?\b",texto_html ))``: Hago la longitud de la lista que devuelve ``re.findall(r"\bgraves?\b",texto_html )``. Dada la regex (``r"\bgraves?\b"``):
         1. ``\b`` al principio y al final: Coincidencia en palabra exacta, no vale ``gravedad`` por ejemplo.
         2. ``grave``
         3. ``s?``: opcional.
   4. Genero el fichero de salida.

In [22]:
df_agregado = None  
medicamentos_agregados = copy.deepcopy(medicamentos)
for medicamento_agregado in medicamentos_agregados:
    url_html = medicamento_agregado["url_html_ficha_tecnica"]
    if url_html is not pd.NA:
        requests.get(
            url = url_html,
            headers={
                "User-Agent": "Educational Scraper"
            }
        )
        if res.status_code != 200:
            medicamentos_agregados = []
            break
        soup = BeautifulSoup(res.content, "html.parser")
        seccion4_4 = soup.find(id="4.4")
        if seccion4_4 is not None:
            div = seccion4_4.find_next()
            contenido = div.get_text(" ", strip=True)
            num_palabras = len(contenido.split(" "))
            medicamento_agregado["volumen-informativo-seccion-4"] = num_palabras
        else:
            medicamento_agregado["volumen_informativo_seccion_4"] = 0
        medicamento_agregado["complejidad_desglose_clinico"] = len(soup.find_all("table"))
        texto_html = soup.get_text(" ", strip=True)
        medicamento_agregado["num_graves"] = len(re.findall(r"\bgraves?\b",texto_html ))   
        
if len(medicamentos_agregados) > 0:
    df_agregado = pd.DataFrame(
        data=medicamentos_agregados
    )
    df_agregado.to_excel(OUTPUT_FOLDER / "medicamentos_agregados.xlsx", index=False)

## Paso intermedio
Los tipos de las variables con las que se hace el join deben de ser iguales.

In [23]:
print(type(medicamentos[3]["cn"]))

<class 'str'>


## Catálogo enriquecido
**Paso 0:** Imprimir las columnas del ``df_nomenclator`` para comprobar que tiene las que creo que tiene.

1. Defino la lista en ``cols`` con las columnas que quiero extraer del archivo.
2. Leo el fichero.
3. Cambio la columna ``Código Nacional`` del ``df_nomenclator`` a ``str`` para que coincida con el tipo de ``cn`` del ``df_agregado``.
4. Se hace el join con ``.merge(...)``, que devuelve un ``pd.DataFrame()``. Lo voy a explicar de manera sencilla, aunque quiero que sepáis que realmente no hace falta hacerlo sobre el que quiero mantener si cambio algún parámetro.
   1. Se hace sobre el que se quiere agregar datos (``df_agregado.merge(...)``).
   2. El "derecho" es el que he descargado, del que se extraen los datos para unirlos al izquierdo.
   3. ``how=left``, los cambios se aplican al de la izquierda ``df_agregado`` (que es sobre el que aplico la función merge).
   4. Indicar la columna que se utilizará para hacer el join, tanto del izquierdo como del derecho.
5. Exportar.
6. 
**Este join: "Oye, coge el df de mi derecha, únelo al df de mi izquierda, pero para que los datos se le pongan a las filas correctas, debes saber que la columna  ``Código Nacional`` de mi derecha se corresponde con la columna ``cn`` del df la izquierda."**


**Al hacer left join, mantengo ``cn``y elimino ``Código Nacional``, si hiciese right join, mantendría ``Código Nacional``y eliminaría ``cn``**



In [24]:
cols = ["Código Nacional", "Estado", "Precio venta al público con IVA", "Precio de referencia", "Tratamiento de larga duración", "Especial control médico"]
df_nomenclator = pd.read_csv(DATA_FOLDER / "20261019_Nomenclator_de_Facturacion.csv", usecols=cols)
df_nomenclator["Código Nacional"] = df_nomenclator["Código Nacional"].astype("string")
if df_agregado is not None:
    df_enriquecido = df_agregado.merge(
        right = df_nomenclator,
        how="left",
        left_on="cn",
        right_on="Código Nacional"
    )
    df_enriquecido.to_excel(OUTPUT_FOLDER / "medicamentos_enriquecidos.xlsx", index = False)
    
print(df_enriquecido.isna().sum()   ) 
print(df_enriquecido.shape)

nregistro                            0
nombre                               0
pactivos                             0
labtitular                           0
labcomercializador                   0
cn                                   0
forma_farmaceutica_simplificada      0
estado_aut                           0
estado_rev                         869
vias_administracion                  0
requiere_receta                      0
afecta_conduccion                    0
triangulo_negro                      0
medicamento_huerfano                 0
biosimilar                           0
url_html_ficha_tecnica               0
url_foto_materiales                784
num_registros_atc                    0
num_principios_activos               0
num_excipientes                      0
volumen_informativo_seccion_4        0
complejidad_desglose_clinico         0
num_graves                           0
Código Nacional                    287
Estado                             287
Precio venta al público c